En este Notebook se introduce por primera vez FakeNewsNet.

La función del Notebook es coger del framework de datos un CSV y prepararlo.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
 # Ruta del proyecto en Google Drive
PROJECT_DIR = "/content/drive/MyDrive/TFG-FakeNewsNet"

# Me muevo a la carpeta principal del proyecto
%cd $PROJECT_DIR

!ls

/content/drive/MyDrive/TFG-FakeNewsNet
data  models  notebooks  README.md  requirements.txt  results


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import json

from urllib.parse import urlparse


# Ruta a FakeNewsNet

In [ ]:
FNN_ROOT = "data/fnn_raw/FakeNewsNet-master/dataset"

In [ ]:
# Cargo los CSV de FakeNewsNet
files = {
    "politifact_fake.csv": ("politifact", 1),
    "politifact_real.csv": ("politifact", 0),
    "gossipcop_fake.csv": ("gossipcop", 1),
    "gossipcop_real.csv": ("gossipcop", 0),
}


In [ ]:
# Comprobación de las columnas del CSV
df_test = pd.read_csv(os.path.join(FNN_ROOT, "politifact_fake.csv"))
df_test.columns


Index(['id', 'news_url', 'title', 'tweet_ids'], dtype='object')

In [ ]:
# Extracción inicial de dominio desde la URL (sin normalización avanzada)

def extract_domain(url):
    if not isinstance(url, str):   # Se comprueba que la URL sea un string
        return None

    try:
        if not url.startswith(("http://", "https://")):   # Obligo a que todas las URL empiecen por http:// para que "urlparse" funcione correctamente
            url = "http://" + url

        parsed = urlparse(url)  # Se extrae únicamente el dominio
        return parsed.netloc
    except Exception:
        return None


In [ ]:
dfs = []

URL_COL = "news_url"

for fname, (source, label) in files.items():  # Se recorren los 4 CSV
    path = os.path.join(FNN_ROOT, fname)
    df_tmp = pd.read_csv(path)

    df_tmp["source_dataset"] = source   # Se añade de que CSV viene (Politifact o GossipCop)
    df_tmp["label"] = label   # Se añade si es fake o true
    df_tmp["domain"] = df_tmp[URL_COL].apply(extract_domain)   # Se añade la columna domain (será el nodo del grafo)

    dfs.append(df_tmp)

df_fnn = pd.concat(dfs, ignore_index=True)   # Se concatenan los 4 dataframes en 1 solo

print("Noticias cargadas:", len(df_fnn))
df_fnn.head()


Noticias cargadas: 23196


,id,news_url,title,tweet_ids,source_dataset,label,domain
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,BREAKING: First NFL Team Declares Bankruptcy O...,937349434668498944\t937379378006282240\t937380...,politifact,1,speedtalk.com
1,politifact15156,politics2020.info/index.php/2018/03/13/court-o...,Court Orders Obama To Pay $400 Million In Rest...,972666281441878016\t972678396575559680\t972827...,politifact,1,politics2020.info
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467...,UPDATE: Second Roy Moore Accuser Works For Mic...,929405740732870656\t929439450400264192\t929439...,politifact,1,www.nscdscamps.org
3,politifact14355,https://howafrica.com/oscar-pistorius-attempts...,Oscar Pistorius Attempts To Commit Suicide,886941526458347521\t887011300278194176\t887023...,politifact,1,howafrica.com
4,politifact15371,http://washingtonsources.org/trump-votes-for-d...,Trump Votes For Death Penalty For Being Gay,915205698212040704\t915242076681506816\t915249...,politifact,1,washingtonsources.org


# Normalización de dominios

In [ ]:
def normalize_domain(domain):
    if not isinstance(domain, str):
        return None

    domain = domain.lower().strip()

    if domain.startswith("http://"):   # Quito los esquemas si por alguna razón han entrado en domain
        domain = domain.replace("http://", "")
    if domain.startswith("https://"):
        domain = domain.replace("https://", "")
    if domain.startswith("www."):   # Se quita el prefijo "www."
        domain = domain[4:]
    return domain

df_fnn["domain"] = df_fnn["domain"].apply(normalize_domain)


In [ ]:
before = len(df_fnn)

df_fnn = df_fnn[df_fnn["domain"].notna()]   # Se eliminan las filas con dominio nulo
df_fnn = df_fnn[df_fnn["domain"].str.contains(r"\.")]   # Me quedo solo con los dominios que tienen un solo punto (así descarto dominios inválidos)

after = len(df_fnn)

print(f"Filas eliminadas por dominio inválido: {before - after}")
print(f"Noticias finales: {after}")
print(f"Dominios únicos: {df_fnn['domain'].nunique()}")


Filas eliminadas por dominio inválido: 330
Noticias finales: 22866
Dominios únicos: 2429


In [ ]:
# Renombrar columna de dominio normalizado

df_fnn = df_fnn.rename(columns={"domain": "domain_norm"})


In [ ]:
# Selección del dataset final

df_fnn = df_fnn[
    [
        "id",
        "news_url",
        "domain_norm",
        "label",
        "source_dataset",
    ]
]

df_fnn.head()


,id,news_url,domain_norm,label,source_dataset
0,politifact15014,speedtalk.com/forum/viewtopic.php?t=51650,speedtalk.com,1,politifact
1,politifact15156,politics2020.info/index.php/2018/03/13/court-o...,politics2020.info,1,politifact
2,politifact14745,www.nscdscamps.org/blog/category/parenting/467...,nscdscamps.org,1,politifact
3,politifact14355,https://howafrica.com/oscar-pistorius-attempts...,howafrica.com,1,politifact
4,politifact15371,http://washingtonsources.org/trump-votes-for-d...,washingtonsources.org,1,politifact


In [ ]:
OUTPUT_PATH = "data/fnn_processed.csv"
df_fnn.to_csv(OUTPUT_PATH, index=False)

print(f"Dataset FakeNewsNet procesado guardado en {OUTPUT_PATH}")


Dataset FakeNewsNet procesado guardado en data/fnn_processed.csv
